# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: FlyRank content decay — flagging declining content for editorial refresh.**

**Task type: Classification** (binary — declining vs. not-declining), with the model's output
probability doubling as a priority score for the editor's queue.

This is a classification problem, not clustering (we're not discovering unlabeled groups — we
already have an observed outcome per item) and not pure ranking (there's no set of items
competing for a fixed number of slots against each other; each content item is judged on its
own). It behaves like a *scoring* problem downstream — editors work a ranked queue — but the
label itself is a yes/no: did this content's performance decline over the trailing window or not.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.shape)
df["trend_direction"].value_counts(dropna=False)


(30000, 44)


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining` = 1 if `trend_direction == "down"`, else 0.**

`trend_direction` is computed from `trend_pct`, which compares each content item's clicks over
the last 30 days against the prior 30 days. This is an **observed** outcome measured directly
from traffic logs, not a human-defined rule (nobody decided "declining" means X in isolation —
it falls out of an actual before/after comparison in the data).

Important constraint from the data skill: **`trend_pct` and `trend_direction` can never be
features.** The label is derived from them, so using either as an input would let the model
"predict" the label from the exact number that built it — circular, not learning. Every other
column (search volume, position, engagement, freshness, age, content type, etc.) is fair game
as a feature; these two are held out entirely.


In [2]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

label_source_cols = ["trend_direction", "trend_pct"]  # never used as features — label leakage
feature_cols = [c for c in df.columns if c not in label_source_cols + ["is_declining", "content_id", "client_id"]]

print("base rate (share declining):", round(df["is_declining"].mean(), 3))
print("n candidate feature columns:", len(feature_cols))


base rate (share declining): 0.542
n candidate feature columns: 40


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: precision@K** (K = size of the editor's actual weekly refresh queue,
e.g. top 100 or top 10% of content by predicted decline probability), with **ROC-AUC** as a
secondary, threshold-free check during model comparison.

Why not plain accuracy: the base rate of "declining" is already ~54%, so a model that just
predicts "declining" for everything scores 54% accuracy without learning anything — that number
is nearly meaningless here. What actually matters for the decision is: **of the content items
we send editors to refresh this week, how many were actually declining?** That's precision at
the top of the queue, since editor time is the scarce resource, not raw classification
correctness on all 30,000 rows.


In [3]:
k = int(0.10 * len(df))  # top 10% of items, roughly one editorial sprint's worth
naive_all_declining_precision = df["is_declining"].mean()  # baseline: precision if we just guess "declining" everywhere

print(f"queue size (K) for a 10% sprint: {k}")
print(f"naive baseline precision (predict all declining): {naive_all_declining_precision:.3f}")
print("-> any real model must beat this at top-K to be worth deploying")


queue size (K) for a 10% sprint: 3000
naive baseline precision (predict all declining): 0.542
-> any real model must beat this at top-K to be worth deploying


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one piece of published content** (`content_id`), belonging to one client
(`client_id`), summarizing its trailing-90-day search performance. 30,000 rows, 32 clients.


In [4]:
unit_of_analysis_cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "avg_position", "impressions_90d", "clicks_90d", "ctr",
    "trend_pct", "trend_direction", "is_declining",
]
df[unit_of_analysis_cols].head(10)


,content_id,client_id,content_type,content_age_days,avg_position,impressions_90d,clicks_90d,ctr,trend_pct,trend_direction,is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,187,10.6,3803,29,0.76,-41.4,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,20.3,15320,7,0.05,-57.7,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,36.5,12581,11,0.09,-60.9,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,6.2,11751,58,0.49,-13.8,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,44.0,19140,24,0.13,-34.7,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,8.5,3970,1,0.03,-38.9,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,7.0,20,0,0.00,-92.3,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,21.2,1724,1,0.06,0.6,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,46.0,32574,29,0.09,-58.8,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,4.9,1240,2,0.16,-29.2,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A single-signal (or even two-signal) if-statement doesn't separate decliners cleanly here.**

Below, no individual tier (search position, content age, freshness) comes close to a clean
split — every tier has both "up" and "down" content mixed in, roughly 40-65% either way. A
concrete rule combining two intuitive signals ("stale AND poorly ranked") actually does *worse*
than the naive "always guess declining" baseline. The signal is real (some combinations of
position, freshness, content type, and engagement do correlate with decline) but it's spread
thinly across ~40 tangled, weakly-informative columns — exactly the case where ML earns its
keep instead of a dashboard or an if-statement, per the framing skill's rule: *"the pattern is
real but too messy to write by hand."*


In [5]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

# does search-position tier alone separate decliners?
print("share declining by position_tier:")
print(df.groupby("position_tier")["is_declining"].mean().round(2))
print()

# a plausible two-signal rule: flag as declining if stale AND badly ranked
rule_pred = (
    (df["avg_position"] > df["avg_position"].median()) &
    (df["content_age_days"] > df["content_age_days"].median())
).astype(int)

print("two-signal rule  -> precision:", round(precision_score(df["is_declining"], rule_pred), 3),
      "| recall:", round(recall_score(df["is_declining"], rule_pred), 3),
      "| accuracy:", round(accuracy_score(df["is_declining"], rule_pred), 3))
print("naive baseline   -> accuracy (predict all declining):", round(df["is_declining"].mean(), 3))
print()
print("The hand-written rule UNDERPERFORMS the naive baseline -> single/paired signals aren't enough.")


share declining by position_tier:
position_tier
deep        0.34
page_1      0.57
page_3_5    0.56
striking    0.61
top_3       0.24
Name: is_declining, dtype: float64

two-signal rule  -> precision: 0.462 | recall: 0.214 | accuracy: 0.439
naive baseline   -> accuracy (predict all declining): 0.542

The hand-written rule UNDERPERFORMS the naive baseline -> single/paired signals aren't enough.
